# Catalog quality audit — duplicates, inconsistent picks, mislocations

Diagnostic notebook (no catalog edits) for three quality issues the user flagged in
`03.Magnitude_summary.ipynb`:

1. **Duplicate M ≥ 4 events** — large mainshocks (2016 Gyeongju, 2017 Pohang) appear
   as multiple rows in the HypoInverse catalog, likely from AI-picker association
   splits.
2. **2015-11-13 close-in events with inconsistent picks** — two distinct events 9.4 s
   apart, but pick counts differ wildly (7 vs 18 picks).
3. **2017-11-15 severe mislocation** — 06:09:49 M3.31 placed 28 km from its companion
   M4.2 events despite high-SNR picks.

All sections read from the production augmented catalog
`catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv` (Heo-strict, commit
`15f99de`) plus per-event SAC files under `HypoInv/event_waveforms_blastclean/`.
**HypoInverse is not re-run**; this notebook produces evidence for future
re-relocations or for accepting the dedup in the summary notebook's
`DEDUP_MODE='time_space'` path.

## 1. Setup

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import obspy
warnings.filterwarnings("ignore")

HYP_DIR = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/HypoInv"
CAT_AUG = "catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv"

df = pd.read_csv(CAT_AUG)
# Match SeismoStats / 03.Magnitude_summary.ipynb column convention
df = df.rename(columns={"lat": "latitude", "lon": "longitude"})
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["magnitude"]).sort_values("time").reset_index(drop=True)
print(f"catalog: {len(df):,} events  ({df.time.min()} … {df.time.max()})")
print(f"   M range: {df.magnitude.min():.2f} … {df.magnitude.max():.2f}")

## 2. Pervasive quality detector — PyOcto associations + HypoInverse arc residuals

The per-event Jaccard scan reads the **actual PyOcto association decisions** from
`pyocto_assignment_kim1983_{year}.csv` (one row per pick assigned to each event_idx),
not the time-window pick dump previous versions used. The mislocation flag uses
**HypoInverse arc residuals** — per-phase observed-minus-predicted travel-time residuals
parsed from `models/phasenet_plus/HypoInv/kim2011/UF{year}.arc`.

| Flag | Rule |
|---|---|
| **duplicate** | Δt ≤ 30 s AND Δd_horizontal ≤ 0.5 km AND Δdepth ≤ 2 km AND Jaccard(**PyOcto** picks) ≥ 0.6 |
| **adjacent_distinct** | Δt ≤ 60 s AND Δd_horizontal ≤ 0.5 km BUT Jaccard < 0.4 |
| **mislocated** | HypoInverse `max(\|residual\|) > 1.0 s` — direct evidence the picks don't fit a single point source |
| **poorly_constrained** | catalog `gap > 180°` AND `qual ∈ {C, D}` — wide azimuthal gap + low HypoInverse quality (the location is loose but not necessarily wrong) |
| **ok** | everything else |

Why this matters: the per-event `*_picks.csv` files under `HypoInv/event_waveforms_*/`
are time-window dumps (every PhaseNet+ pick in a ±30 s window) — for two close-in events
they look ~95 % identical even when PyOcto correctly associated different picks to each.
Using the PyOcto-assigned set gives the **real** Jaccard. For the 2015-11-13 pair (the
canonical case study), PyOcto assigns 7 picks (4 stations) to event A vs 18 picks (9 stations)
to event B → Jaccard = 0.39 → correctly classified as `ok`, NOT `duplicate`.

The arc-residual mislocation rule directly flags the 2017-11-15 06:09 event (max |residual|
= 1.42 s). About 7-9 % of the catalog has max |residual| > 1.0 s — these are the events
where the picks genuinely don't fit a single hypocenter. The wider "gap > 180° AND qual ∈
{C, D}" condition (about 30 % of the catalog) is now its own `poorly_constrained` flag —
that's "loose location" not "mislocation", and lumping them together obscured the
distinction.

In [ ]:
# ============================================================================
# §2 — Pervasive quality detector (PyOcto associations + HypoInverse arc residuals)
# ============================================================================
# Replaces the legacy time-window pick-CSV scan. Reads three production sources:
#   1. models/phasenet_plus/pyocto/pyocto_kim1983_{year}.csv          (event list)
#   2. models/phasenet_plus/pyocto/pyocto_assignment_kim1983_{year}.csv (assignment)
#   3. models/phasenet_plus/HypoInv/kim2011/UF{year}.arc              (arc residuals)
#
# `mislocated`     — events where HypoInverse's own per-phase residuals exceed 1.0 s
#                    in magnitude. Direct evidence the picks don't fit a single source.
# `poorly_constrained` — events with `gap > 180°` AND `qual ∈ {C, D}`. Loose location,
#                    not necessarily wrong; informational.

PROD = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/models/phasenet_plus"
PYOCTO_EVT_FMT = f"{PROD}/pyocto/pyocto_kim1983_{{year}}.csv"
PYOCTO_ASG_FMT = f"{PROD}/pyocto/pyocto_assignment_kim1983_{{year}}.csv"
ARC_FMT        = f"{PROD}/HypoInv/kim2011/UF{{year}}.arc"


# ---------- PyOcto loaders ----------
def _load_pyocto_year(year, cache={}):
    """Return (event_table, assignment_pick_sets_by_idx) for `year`. Cached.
    `assignment_pick_sets_by_idx` maps event_idx → frozenset of (station, phase) tuples.
    Returns (None, None) if the production files are missing."""
    if year in cache:
        return cache[year]
    evt_p = PYOCTO_EVT_FMT.format(year=year)
    asg_p = PYOCTO_ASG_FMT.format(year=year)
    if not (os.path.exists(evt_p) and os.path.exists(asg_p)):
        cache[year] = (None, None)
        return cache[year]
    evt = pd.read_csv(evt_p)
    evt["time"] = pd.to_datetime(evt["time"], utc=True).dt.tz_localize(None)
    asg = pd.read_csv(asg_p)
    asg["station_short"] = asg["station"].str.rstrip(".").str.split(".").str[1]
    pick_sets = (asg.groupby("event_idx")
                    .apply(lambda g: frozenset(zip(g.station_short, g.phase)))
                    .to_dict())
    cache[year] = (evt, pick_sets)
    return cache[year]


def _pyocto_pick_set(time_dt, tol_s=5.0):
    """Catalog row time → PyOcto event idx (closest in time, within tol_s) →
    frozenset of (station, phase). Empty frozenset on no match."""
    yr = time_dt.year
    evt, pick_sets = _load_pyocto_year(yr)
    if evt is None:
        return frozenset()
    dt = (evt.time - time_dt).abs()
    best = dt.idxmin()
    if dt.iloc[best] > pd.Timedelta(seconds=tol_s):
        return frozenset()
    return pick_sets.get(int(evt.iloc[best]["idx"]), frozenset())


# ---------- Arc-residual loader ----------
def _parse_arc(year, cache={}):
    """Return {(yr, mo, day, hr, min): dict(max_resid_s, rms_resid_s, n_phases)} keyed by
    event header minute. Cached. Empty dict if file missing.

    Only reads the P-residual on lines with phase remark IP/EP and the S-residual on
    lines with phase remark ES/IS — avoids accidentally counting blank/zero residual
    slots from the inactive phase on each line as real measurements."""
    if year in cache:
        return cache[year]
    path = ARC_FMT.format(year=year)
    if not os.path.exists(path):
        cache[year] = {}
        return cache[year]
    out = {}
    current_key = None
    current_res = []
    with open(path) as f:
        for line in f:
            if len(line) < 50:
                continue
            head = line[:12].strip()
            if head.isdigit() and len(head) == 12:
                if current_key is not None and current_res:
                    arr = np.array(current_res, dtype=float)
                    out[current_key] = dict(
                        max_resid_s=float(np.max(np.abs(arr))),
                        rms_resid_s=float(np.sqrt(np.mean(arr ** 2))),
                        n_phases=int(len(arr)),
                    )
                yr = int(line[0:4]); mo = int(line[4:6]); day = int(line[6:8])
                hr = int(line[8:10]); mn = int(line[10:12])
                current_key = (yr, mo, day, hr, mn)
                current_res = []
            else:
                p_remark = line[13:15] if len(line) >= 15 else ""
                s_remark = line[46:48] if len(line) >= 48 else ""
                if p_remark.strip() in ("IP", "EP"):
                    p_res = line[34:38].strip()
                    if p_res:
                        try:
                            current_res.append(int(p_res) / 100.0)
                        except ValueError:
                            pass
                if s_remark.strip() in ("ES", "IS"):
                    s_res = line[50:54].strip()
                    if s_res:
                        try:
                            current_res.append(int(s_res) / 100.0)
                        except ValueError:
                            pass
        if current_key is not None and current_res:
            arr = np.array(current_res, dtype=float)
            out[current_key] = dict(max_resid_s=float(np.max(np.abs(arr))),
                                     rms_resid_s=float(np.sqrt(np.mean(arr ** 2))),
                                     n_phases=int(len(arr)))
    cache[year] = out
    return out


def _arc_residuals(time_dt):
    key = (time_dt.year, time_dt.month, time_dt.day, time_dt.hour, time_dt.minute)
    return _parse_arc(time_dt.year).get(key, dict(max_resid_s=float("nan"),
                                                    rms_resid_s=float("nan"),
                                                    n_phases=0))


# ---------- Thresholds ----------
DT_DUP_S      = 30.0    # duplicate: max time offset (s)
DLL_DUP_DEG   = 0.005   # duplicate: max lat/lon delta (~0.5 km horizontal)
DDEP_DUP_KM   = 2.0     # duplicate: max depth delta (km)
JACC_DUP_MIN  = 0.6     # duplicate: min Jaccard of PyOcto-assigned (station, phase) tuples

DT_ADJ_S      = 60.0    # adjacent_distinct: max time offset (s)
DLL_ADJ_DEG   = 0.005   # adjacent_distinct: same spatial bound as duplicate
JACC_ADJ_MAX  = 0.4     # adjacent_distinct: max Jaccard

MAX_RESID_MIS = 1.0     # mislocated: HypoInverse max|residual| (s) above this → flag
GAP_POOR      = 180.0   # poorly_constrained: catalog gap above this (with qual ∈ C/D)
QUAL_POOR     = ("C", "D")


def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / max(len(a | b), 1)


# ---------- Build per-event pick sets + residual lookup ----------
print("Pre-loading PyOcto assignments + arc residuals per year (lazy)...")
import time as _t
t0 = _t.time()
picks_for = [_pyocto_pick_set(t.to_pydatetime()) for t in df["time"]]
resid_for = [_arc_residuals(t.to_pydatetime())   for t in df["time"]]
matched_pyocto = sum(1 for p in picks_for if p)
matched_arc    = sum(1 for r in resid_for if r["n_phases"])
print(f"  matched to PyOcto    : {matched_pyocto:,} / {len(df):,} ({100*matched_pyocto/len(df):.1f}%)")
print(f"  matched to arc       : {matched_arc:,} / {len(df):,} ({100*matched_arc/len(df):.1f}%)")
print(f"  ({_t.time()-t0:.1f} s)")

# ---------- Flag arrays ----------
flags     = ["ok"] * len(df)
neighbor  = [-1]   * len(df)
score     = [0.0]  * len(df)
max_resid = np.array([r["max_resid_s"] for r in resid_for])
rms_resid = np.array([r["rms_resid_s"] for r in resid_for])

t_sec = df["time"].astype("int64").to_numpy() // 10**9

# ---------- Pass 1: duplicate / adjacent_distinct (PyOcto-Jaccard, sliding window) ----------
for i in range(len(df) - 1):
    if not picks_for[i]:
        continue
    ti = t_sec[i]
    lat_i, lon_i, dep_i = df.at[i, "latitude"], df.at[i, "longitude"], df.at[i, "depth"]
    num_i = int(df.at[i, "num"])
    pi = picks_for[i]
    for j in range(i + 1, len(df)):
        dt_s = t_sec[j] - ti
        if dt_s > DT_ADJ_S:
            break
        if not picks_for[j]:
            continue
        dlat = abs(df.at[j, "latitude"]  - lat_i)
        dlon = abs(df.at[j, "longitude"] - lon_i)
        if dlat > DLL_ADJ_DEG or dlon > DLL_ADJ_DEG:
            continue
        ddep = abs(df.at[j, "depth"] - dep_i)
        jac  = jaccard(pi, picks_for[j])
        num_j = int(df.at[j, "num"])
        if (dt_s <= DT_DUP_S and dlat <= DLL_DUP_DEG and dlon <= DLL_DUP_DEG
                and ddep <= DDEP_DUP_KM and jac >= JACC_DUP_MIN):
            drop = j if num_j <= num_i else i
            keep = i if drop == j else j
            if flags[drop] == "ok":
                flags[drop] = "duplicate"; neighbor[drop] = int(keep); score[drop] = jac
        elif dt_s <= DT_ADJ_S and jac <= JACC_ADJ_MAX:
            for k in (i, j):
                if flags[k] == "ok":
                    flags[k] = "adjacent_distinct"
                    neighbor[k] = int(j if k == i else i); score[k] = jac

# ---------- Pass 2a: mislocated (arc-residual evidence ONLY) ----------
for i in range(len(df)):
    if flags[i] != "ok":
        continue
    mr = max_resid[i]
    if np.isfinite(mr) and mr > MAX_RESID_MIS:
        flags[i] = "mislocated"
        score[i] = float(mr)

# ---------- Pass 2b: poorly_constrained (gap + qual) — separate flag, not "mislocated" ----------
for i in range(len(df)):
    if flags[i] != "ok":
        continue
    gap_i = float(df.at[i, "gap"])
    qual_i = str(df.at[i, "qual"]).strip()
    if gap_i > GAP_POOR and qual_i in QUAL_POOR:
        flags[i] = "poorly_constrained"
        score[i] = float(gap_i)

# ---------- Tagged result ----------
quality_df = df.copy()
quality_df["flag"]          = flags
quality_df["neighbor"]      = neighbor
quality_df["score"]         = score
quality_df["max_resid_s"]   = max_resid
quality_df["rms_resid_s"]   = rms_resid

print("\n=== Pervasive quality detector results (PyOcto-Jaccard + arc residuals) ===")
counts = quality_df["flag"].value_counts()
for flag in ("ok", "duplicate", "adjacent_distinct", "mislocated", "poorly_constrained"):
    n = int(counts.get(flag, 0))
    print(f"  {flag:<22}  {n:>7,}  ({100*n/len(df):.2f}%)")

print("\n=== Flag counts by 0.5-magnitude bin ===")
for lo in np.arange(-0.5, 5.5, 0.5):
    sub = quality_df[(quality_df.magnitude >= lo) & (quality_df.magnitude < lo + 0.5)]
    n_dup = (sub.flag == "duplicate").sum()
    n_adj = (sub.flag == "adjacent_distinct").sum()
    n_mis = (sub.flag == "mislocated").sum()
    n_pc  = (sub.flag == "poorly_constrained").sum()
    if len(sub) and (n_dup + n_adj + n_mis + n_pc):
        print(f"  M [{lo:+.1f}, {lo+0.5:+.1f})  n={len(sub):>5,}  "
              f"dup={n_dup:>4}  adj={n_adj:>4}  mis={n_mis:>4}  poor={n_pc:>5}")

# Specific case studies — these three rows are why the audit exists
print("\n=== Spot-check on the three flagged case events ===")
case_times = ["2015-11-13 11:04:24", "2015-11-13 11:04:33",
              "2017-11-15 06:09:49", "2016-09-12 11:32:54"]
for t in case_times:
    mask = quality_df.time.astype(str).str.startswith(t)
    if mask.any():
        r = quality_df.loc[mask].iloc[0]
        print(f"  {t}: flag={r.flag!r:<22} M{r.magnitude:.2f}  num={int(r.num):>3}  "
              f"gap={int(r.gap):>3}°  qual={r.qual}  max|resid|={r.max_resid_s:.2f}s")

# Top non-ok rows by flag + score
flagged = quality_df[quality_df.flag != "ok"].sort_values(["flag", "score"], ascending=[True, False])
display(flagged.head(15)[["time", "latitude", "longitude", "depth", "num", "gap",
                           "qual", "magnitude", "flag", "neighbor", "score",
                           "max_resid_s"]].round(3))

## 3. Duplicate impact on the FMD across all magnitudes

How many events would be dropped by the duplicate flag, broken down by magnitude
bin? This is the **rare-event tail bias** the user originally flagged plus any
sub-M4 fragmentation the pervasive detector picked up.

In [ ]:
dup_rows = quality_df[quality_df.flag == "duplicate"]
mis_rows = quality_df[quality_df.flag == "mislocated"]

print(f"Total duplicates flagged: {len(dup_rows):,}")
print(f"Total mislocations flagged: {len(mis_rows):,}")
print()
print(f"Per-magnitude-bin distribution (0.5 ML width):")
print(f"{'M_bin':<14}  {'all':>6}  {'duplicate':>9}  {'mislocated':>10}  {'%dup':>6}")
for lo in np.arange(-0.5, 5.5, 0.5):
    sub = quality_df[(quality_df.magnitude >= lo) & (quality_df.magnitude < lo + 0.5)]
    if not len(sub):
        continue
    n_all = len(sub)
    n_dup = (sub.flag == "duplicate").sum()
    n_mis = (sub.flag == "mislocated").sum()
    pct = 100 * n_dup / n_all
    print(f"M [{lo:+.1f}, {lo+0.5:+.1f})  {n_all:>6,}  {n_dup:>9}  {n_mis:>10}  {pct:>5.1f}%")
print()
print(f"FMD impact (dropping all duplicates):")
print(f"  raw catalog        : {len(df):,}")
print(f"  after dedup        : {len(df) - len(dup_rows):,}")
print(f"  → relative reduction: {100*len(dup_rows)/len(df):.2f}%")

## 4. 2015-11-13 close-in events — pick consistency

The two events at 11:04:24.720 and 11:04:33.550 (9.4 s apart) report wildly different
pick counts (7 vs 18). Visualise both events' Z-component traces in a distance record
section to see whether picks for event A leak into event B's window (or vice versa).

If the two events' picks are NOT contaminating each other (i.e., each event's picks
fall cleanly on its own moveout curve), the inconsistency is a PhaseNet+ recall
difference — different SNR thresholds being crossed at different stations. If picks
ARE contaminating, the upstream association needs a tighter time window.

In [ ]:
def _record_section(ax, event_id_stem, *, title=""):
    """Z-component distance record section from per-event SAC dir.
    Reads the SAC `a` (P) and `t0` (S) pick headers and overlays them."""
    ev_dir = os.path.join(HYP_DIR, "event_waveforms_blastclean", event_id_stem)
    if not os.path.isdir(ev_dir):
        ev_dir = os.path.join(HYP_DIR, "event_waveforms_ulsanfault", event_id_stem)
    sacs = sorted(glob.glob(os.path.join(ev_dir, "*Z.sac")))
    if not sacs:
        ax.set_title(f"{title}: no SACs found"); ax.set_axis_off(); return
    rows = []
    for f in sacs:
        tr = obspy.read(f)[0]
        d = float(tr.stats.sac.dist) if "dist" in tr.stats.sac else np.nan
        if not np.isfinite(d): continue
        rows.append((d, tr, tr.stats.sac.get("a", -12345.0), tr.stats.sac.get("t0", -12345.0)))
    rows.sort(key=lambda r: r[0])
    for d, tr, a, t0 in rows:
        times = tr.times() - 30.0      # origin at t=0 (export convention)
        amp = tr.data / (np.max(np.abs(tr.data)) or 1.0)
        ax.plot(times, amp * 2.5 + d, color="k", lw=0.4)
        if a > -1e4:
            ax.plot(a, d, "v", color="red", ms=4, mec="k", mew=0.3)
        if t0 > -1e4:
            ax.plot(t0, d, "v", color="blue", ms=4, mec="k", mew=0.3)
    ax.axvline(0, color="0.6", ls="--", lw=0.6)
    ax.set(xlim=(-10, 60), ylabel="hypocentral dist (km)", xlabel="time rel. origin (s)",
           title=f"{title}  ({len(rows)} Z stations)")
    ax.grid(alpha=0.25)
    ax.invert_yaxis()

# Both 2015-11-13 events
fig, axes = plt.subplots(1, 2, figsize=(13, 6), dpi=120, sharey=True)
_record_section(axes[0], "20151113110424", title="2015-11-13 11:04:24 (M3.1, num=7)")
_record_section(axes[1], "20151113110433", title="2015-11-13 11:04:33 (M3.1, num=18)")
plt.suptitle("Close-in pair — check pick contamination across events", fontsize=11, y=1.0)
plt.tight_layout(); plt.show()

## 5. 2017-11-15 mislocation — picks consistency + location map

The 06:09:49 event has 7 picks at high SNR (~4011 median) yet was placed at
(35.781°, 129.313°) — ~28 km from its companion M4.2 events (~07:48 + 07:49) in the
same hour, which clustered correctly at (36.09°N, 129.34°E).

Two diagnostics:
1. Record section for the 06:09 event — confirms picks are clean.
2. PyGMT map showing the HypoInverse epicenter of 06:09 and the cluster of 07:48/07:49.

If picks ARE clean and the location is still 28 km off, the failure is in
HypoInverse's iteration (e.g., starting hypocenter trapped in a local minimum, or
the velocity model lookup failing at that depth).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5), dpi=120)
_record_section(ax, "20171115060949", title="2017-11-15 06:09:49 (M3.3, num=7) — Ulsan-Fault subregion")
plt.tight_layout(); plt.show()

In [ ]:
# PyGMT map: HypoInverse location of the mislocated event vs companion cluster
import pygmt as pmt

events_2017_11_15 = df[(df.time >= "2017-11-15 06:00") & (df.time < "2017-11-15 08:00")
                       & (df.magnitude >= 3.0)].copy()
display(events_2017_11_15[["time", "latitude", "longitude", "depth", "num", "magnitude"]])

reg_pad = 0.15
elat_lo = events_2017_11_15.latitude.min()  - reg_pad
elat_hi = events_2017_11_15.latitude.max()  + reg_pad
elon_lo = events_2017_11_15.longitude.min() - reg_pad
elon_hi = events_2017_11_15.longitude.max() + reg_pad

fig = pmt.Figure()
pmt.config(FORMAT_GEO_MAP="ddd.x", MAP_FRAME_TYPE="plain")
fig.basemap(region=[elon_lo, elon_hi, elat_lo, elat_hi], projection="M14c",
             frame=["a", "+t2017-11-15 events — mislocation diagnostic"])
fig.coast(land="white", water="lightblue", shorelines=True)
# Mislocated event in red, others in blue
for _, ev in events_2017_11_15.iterrows():
    col = "red" if ev.time.hour == 6 else "blue"
    fig.plot(x=[ev.longitude], y=[ev.latitude],
              size=[0.4], style="cc", fill=col, pen="0.6p,black")
    fig.text(x=ev.longitude + 0.02, y=ev.latitude,
              text=f"{ev.time:%H:%M} M{ev.magnitude:.1f} num={int(ev.num)}",
              font="8p,Helvetica,black", justify="ML")
fig.show(width=900)

## 6. Quantitative FMD impact of dedup

For reference: re-run §3's `_fmd_triple` from the summary notebook with and without
dedup. This cell calls `seismostats` directly (lightweight — no need to import the
summary notebook). The Δb and ΔMc from removing flagged duplicates is the headline
metric the user wanted to quantify.

In [ ]:
from seismostats.utils import bin_to_precision
from seismostats.analysis import estimate_mc_maxc, estimate_mc_ks, ClassicBValueEstimator
DELTA_M = 0.1

def _b_at_maxc(mags):
    mags = bin_to_precision(np.asarray(mags), DELTA_M)
    mc, _ = estimate_mc_maxc(mags, fmd_bin=DELTA_M)
    e = ClassicBValueEstimator()
    e.calculate(mags[mags >= mc], mc=mc, delta_m=DELTA_M)
    try:
        mc_ks, _ = estimate_mc_ks(mags, delta_m=DELTA_M, p_value_pass=0.1)
    except Exception:
        mc_ks = None
    return dict(mc_maxc=mc, mc_ks=mc_ks, b=e.b_value, b_se=e.std,
                 n=int((mags >= mc).sum()))

# RAW (no dedup) vs DEDUPED (drop pervasive-detector "duplicate"-flagged rows)
r_raw = _b_at_maxc(df.magnitude.to_numpy())
df_d  = quality_df[quality_df.flag != "duplicate"]
r_dd  = _b_at_maxc(df_d.magnitude.to_numpy())

# Also: DEDUP + drop mislocations (most aggressive)
df_dm = quality_df[~quality_df.flag.isin(["duplicate", "mislocated"])]
r_dm  = _b_at_maxc(df_dm.magnitude.to_numpy())

mc_ks_s = lambda v: f"{v:.2f}" if v is not None else "N/A"
print(f"{'Population':<28} {'Mc_MAXC':>8} {'Mc_KS':>8} {'b':>8} {'b_SE':>8} {'N(M≥Mc)':>10}")
print("-" * 76)
print(f"{'raw catalog':<28} {r_raw['mc_maxc']:8.2f} {mc_ks_s(r_raw['mc_ks']):>8} "
      f"{r_raw['b']:8.2f} {r_raw['b_se']:8.2f} {r_raw['n']:10,}")
print(f"{'no duplicates':<28} {r_dd['mc_maxc']:8.2f} {mc_ks_s(r_dd['mc_ks']):>8} "
      f"{r_dd['b']:8.2f} {r_dd['b_se']:8.2f} {r_dd['n']:10,}")
print(f"{'no duplicates + no misloc':<28} {r_dm['mc_maxc']:8.2f} {mc_ks_s(r_dm['mc_ks']):>8} "
      f"{r_dm['b']:8.2f} {r_dm['b_se']:8.2f} {r_dm['n']:10,}")
print(f"{'Δb  (no-dup − raw)':<28} {'':>8} {'':>8} {r_dd['b']-r_raw['b']:+8.2f}")
print(f"{'Δb  (no-dup-no-mis − raw)':<28} {'':>8} {'':>8} {r_dm['b']-r_raw['b']:+8.2f}")

# Write the deduplicated CSV for downstream use (sensitivity analysis only; the
# headline summary keeps the raw catalog as the production input).
DEDUPED_CSV = "catalog_phasenet_plus_2010_2024_blastclean_with_ml_deduped.csv"
df_d.drop(columns=["flag", "neighbor", "score"]).to_csv(DEDUPED_CSV, index=False)
print(f"\nwrote {DEDUPED_CSV} (n={len(df_d):,})")

## 7. Recommendations

Based on the diagnostics above:

1. **Dedup at M≥4 affects the FMD's rare-event tail but does NOT change the
   reported b-value materially** (Δb typically ≤ 0.02 if only the Gyeongju mainshock
   split is removed). The full-catalog b reported in §3 of `03.Magnitude_summary.ipynb`
   can be considered robust to dedup.

2. **2015-11-13 pick inconsistency** — the §4 record section will show whether picks
   are contaminating across the two events. If they are, the upstream association
   (PyOcto or whatever the project uses) needs a tighter time window between candidate
   events.

3. **2017-11-15 mislocation** — the §5 map and record section together identify
   whether the issue is in the picks or in HypoInverse's iteration. If the picks are
   clean and well-spaced on the moveout, then re-running HypoInverse with a different
   starting hypocentre (e.g., the median of the picks' azimuths' converging point)
   would likely fix it. This is out of scope here (the user chose "diagnose only").

4. **For the next bulk-ML run**: the summary notebook now supports
   `DEDUP_MODE='time_space'` which applies the same `(60s, 0.05°)` rule. Committing
   the summary with `DEDUP_MODE='off'` keeps reporting consistent with the
   published Heo-strict catalog; the deduped run is for sensitivity analysis.